This notebook contains a script to query the VAST pipeline to get all point like sources within a degree of each pulsar. We use these as candidates to be control sources, which will later be used to make corrected fluxes for each pulsar.

IN: 'paper_dfv2.csv'

OUT: 'all_candidate_controls.csv'

In [1]:
# importing required modules
from vasttools.query import Query
import pandas as pd

from vasttools.moc import VASTMOCS
from astropy import units as u

import numpy as np

from astropy.coordinates import SkyCoord

import scipy.stats as stats

In [2]:
psr_df = pd.read_csv('paper_dfv2.csv')

In [6]:
#filling the below dictionary with all possible candidates for control sources.
cand_ctrls = {}
for name in psr_df['JNAME'].values:
    psr_name = name
    
    #this query of the VAST dataset looks within epoch 23, the first full survey epoch
    query_coord = SkyCoord(str(psr_df[psr_df['JNAME']==psr_name].iloc[0]['RAJD']) + " " + str(psr_df[psr_df['JNAME']==psr_name].iloc[0]['DECJD']), unit='deg')
    query = Query(coords=query_coord, epochs='23', crossmatch_radius=3600., search_around_coordinates=True, use_tiles=True, corrected_data=False)
    query.find_sources()
    
    num_rows_unmasked = query.results.shape[0]
    point_source_mask = query.results['flux_int'] < (1.5 * query.results['flux_peak'])    
    num_rows_masked = query.results[point_source_mask].shape[0]
    
    cand_ctrls[psr_name] = query.results[point_source_mask]
    
    print(name)
    print("        " + str(num_rows_unmasked) + " candidates")
    print("        " + str(num_rows_masked) + " point source candidates")

J1644-4559
        2227 candidates
        1114 point source candidates
J1752-2806
        752 candidates
        532 point source candidates
J0908-4913
        2352 candidates
        1780 point source candidates
J1534-5334
        1039 candidates
        832 point source candidates
J1745-3040
        1635 candidates
        960 point source candidates
J1605-5257
        1418 candidates
        875 point source candidates
J1804-2717
        1967 candidates
        1384 point source candidates
J1604-4909
        2429 candidates
        1877 point source candidates
J1428-5530
        2771 candidates
        2108 point source candidates
J1001-5507
        2695 candidates
        2063 point source candidates
J1722-3207
        2615 candidates
        1857 point source candidates
J1806-1154
        2645 candidates
        2019 point source candidates
J0907-5157
        3135 candidates
        2444 point source candidates
J1717-4054
        1511 candidates
        1224 point source candidat

In [7]:
#creating a dataframe of all the candidates and saving them to a csv
vals = []
names = list(cand_ctrls.keys())
for name in names:
    vals.append(cand_ctrls[name])
candidates_df = pd.concat(vals, keys=names)

candidates_df.to_csv('all_candidate_controls.csv')